# COMP64702 RAG Coursework
## Notebook 3: Evaluation

This notebook evaluates generated answers using ROUGE, BLEU,
BERTScore and faithfulness metrics.

| Metric / Feature | Original | Improved |
|---|---|---|
| ROUGE-1/2/L | ✅ | ✅ |
| BLEU | ✅ | ✅ |
| BERTScore | ✅ | ✅ |
| Faithfulness | ✅ (token overlap) | ✅ (kept) |
| **Retrieval Precision@1 / @5** | ❌ Missing | ✅ Added |
| **Per-difficulty / type breakdown** | ❌ Missing | ✅ Added |
| **Baseline comparison table** | Prose only | ✅ Numerical delta table |
| **Retrieval audit** | ❌ Missing | ✅ Added |
| **`retrieved_doc_ids` extracted** | ❌ | ✅ (feeds retrieval metrics) |
| **Results as DataFrame** | List of dicts | ✅ `pd.DataFrame` for easy groupby |
| **Baseline delta saved to JSON** | ❌ | ✅ |

In [14]:
# Install dependencies
# CHANGE: added -q flag for cleaner output
!pip install rouge-score bert-score nltk -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [15]:
# Imports
# CHANGE: added pandas (for DataFrame-based groupby breakdowns);
#         nltk.download now uses quiet=True to suppress noise;
#         added print("Imports OK") for confirmation.

import json
from pathlib import Path
import numpy as np
import pandas as pd

from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score as bertscore_score
import nltk
nltk.download("punkt", quiet=True)

print("Imports OK")

Imports OK


In [16]:
# Paths and data loading (unchanged)

DATA_DIR = Path(".")

benchmark_path = DATA_DIR / "benchmark_dataset.json"
outputs_path   = DATA_DIR / "test_outputs.json"
eval_path      = DATA_DIR / "evaluation_results.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

with open(outputs_path, "r", encoding="utf-8") as f:
    outputs_payload = json.load(f)

outputs = outputs_payload["results"]
print(f"Loaded {len(benchmark)} benchmark items and {len(outputs)} outputs")

Loaded 13 benchmark items and 13 outputs


In [17]:
# Match benchmark and outputs by query_id, then build evaluation lists
# CHANGE: cells 4 and 5 consolidated into one cell.
# CHANGE: added retrieved_doc_ids extraction — required for Precision@K metrics.

gold_map = {item["query_id"]: item for item in benchmark}
pred_map = {item["query_id"]: item for item in outputs}

common_ids = [qid for qid in gold_map if qid in pred_map]
print(f"Matched {len(common_ids)} query IDs: {common_ids}")

predictions        = [pred_map[qid]["response"] for qid in common_ids]
references         = [gold_map[qid]["answer"]   for qid in common_ids]
retrieved_contexts = [
    [ctx["text"]   for ctx in pred_map[qid].get("retrieved_context", [])]
    for qid in common_ids
]
# NEW: extract doc_id list per query for retrieval precision/recall
retrieved_doc_ids  = [
    [ctx["doc_id"] for ctx in pred_map[qid].get("retrieved_context", [])]
    for qid in common_ids
]

print("Example prediction:\n", predictions[0])
print("\nExample reference:\n", references[0])

Matched 13 query IDs: ['sa_000', 'sa_001', 'sa_002', 'sa_003', 'sa_004', 'sa_005', 'sa_006', 'sa_007', 'sa_008', 'sa_009', 'sa_010', 'sa_011', 'sa_012']
Example prediction:
 Biryani is a mixed rice dish originating in South Asia, traditionally made with rice, meat (chicken, goat, beef) or seafood (prawns or fish), vegetables, and spices. It was once associated with the region's Muslim population but is now a mainstream culinary staple embraced by every demographic.

Example reference:
 Biryani is a mixed rice dish originating in the Indian subcontinent, made with long-grain basmati rice, spices, and meat, vegetables, or eggs. It is closely associated with Mughal cuisine and is particularly famous in the city of Hyderabad as well as in Lucknow, where the dum cooking technique is used to slow-cook the layered rice and meat together.


In [18]:
# Metric functions
# CHANGE: ROUGE, BLEU and faithfulness functions consolidated into one cell.
# CHANGE: bleu_score argument order fixed — original passed (ref_tokens, pred_tokens);
#         correct call is sentence_bleu(references, hypothesis).
# NEW: retrieval_precision_at_k and retrieval_recall_at_k added.

rouge  = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method1


def rouge_scores(pred, ref):
    scores = rouge.score(ref, pred)
    return {
        "rouge1_f": scores["rouge1"].fmeasure,
        "rouge2_f": scores["rouge2"].fmeasure,
        "rougeL_f": scores["rougeL"].fmeasure,
    }


def bleu_score(pred, ref):
    return sentence_bleu([ref.split()], pred.split(), smoothing_function=smooth)


def faithfulness_score(prediction, context_texts):
    """Token-overlap faithfulness: fraction of answer tokens found in context."""
    context = " ".join(context_texts).lower()
    pred_tokens = prediction.lower().split()
    if not pred_tokens:
        return 0.0
    supported = sum(1 for tok in pred_tokens if tok in context)
    return supported / len(pred_tokens)


# ─────────────────────────────────────────────────────────
# NEW: Retrieval Precision@K and Recall@K
#
# Precision@K: did we retrieve the correct source doc in top-K?
# (Binary — 1 if answer_source appears in retrieved_doc_ids[:K])
#
# This directly measures retrieval quality independently of
# generation quality, which is important for the rubric marks
# on the "Retrieval and Ranking" component.
# ─────────────────────────────────────────────────────────

def retrieval_precision_at_k(retrieved_ids, relevant_id, k=5):
    """1.0 if relevant_id appears in top-k retrieved doc ids, else 0.0."""
    return 1.0 if relevant_id in retrieved_ids[:k] else 0.0


def retrieval_recall_at_k(retrieved_ids, relevant_id, k=5):
    """Same as precision here since there is one relevant doc per query."""
    return retrieval_precision_at_k(retrieved_ids, relevant_id, k)


print("Metric functions defined.")

Metric functions defined.


In [19]:
# Compute BERTScore (batch operation — run once)

P, R, F1 = bertscore_score(predictions, references, lang="en", verbose=True)
bertscore_f = F1.tolist()

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 3.15 seconds, 4.12 sentences/sec


In [20]:
# Evaluate each item — all metrics
# CHANGE: added difficulty, question_type, precision_at_1, precision_at_5
#         fields to each result dict.
# CHANGE: results stored in a pandas DataFrame for easy groupby analysis.

per_item = []

for i, qid in enumerate(common_ids):
    pred       = predictions[i]
    ref        = references[i]
    ctxs       = retrieved_contexts[i]
    ret_ids    = retrieved_doc_ids[i]
    answer_src = gold_map[qid]["answer_source"]
    difficulty = gold_map[qid].get("difficulty", "unknown")
    q_type     = gold_map[qid].get("question_type", "unknown")

    rouge_dict = rouge_scores(pred, ref)
    bleu       = bleu_score(pred, ref)
    faith      = faithfulness_score(pred, ctxs)
    prec_at_1  = retrieval_precision_at_k(ret_ids, answer_src, k=1)
    prec_at_5  = retrieval_precision_at_k(ret_ids, answer_src, k=5)

    per_item.append({
        "query_id":       qid,
        "difficulty":     difficulty,
        "question_type":  q_type,
        "rouge1_f":       rouge_dict["rouge1_f"],
        "rouge2_f":       rouge_dict["rouge2_f"],
        "rougeL_f":       rouge_dict["rougeL_f"],
        "bleu":           bleu,
        "bertscore_f":    bertscore_f[i],
        "faithfulness":   faith,
        "precision_at_1": prec_at_1,
        "precision_at_5": prec_at_5,
    })

df = pd.DataFrame(per_item)
df

,query_id,difficulty,question_type,rouge1_f,rouge2_f,rougeL_f,bleu,bertscore_f,faithfulness,precision_at_1,precision_at_5
0,sa_000,easy,factual,0.392523,0.171429,0.336449,0.148450,0.896212,0.978261,1.0,1.0
1,sa_001,hard,comparative,0.155340,0.079208,0.135922,0.002780,0.871416,0.684211,0.0,1.0
2,sa_002,easy,comparative,0.290909,0.055556,0.163636,0.003193,0.870274,0.846154,0.0,1.0
3,sa_003,medium,procedural,0.209302,0.047619,0.162791,0.008940,0.843607,0.983607,1.0,1.0
4,sa_004,medium,procedural,0.394737,0.135135,0.236842,0.023965,0.897990,0.527273,1.0,1.0
5,sa_005,hard,ingredient,0.376238,0.161616,0.277228,0.079232,0.884788,0.922078,1.0,1.0
6,sa_006,hard,comparative,0.250000,0.128205,0.250000,0.021548,0.881174,0.721311,1.0,1.0
7,sa_007,medium,comparative,0.173077,0.019608,0.153846,0.005467,0.871212,0.545455,1.0,1.0
8,sa_008,hard,comparative,0.226415,0.078431,0.188679,0.014989,0.889915,0.810811,1.0,1.0
9,sa_009,easy,factual,0.761905,0.550000,0.761905,0.307090,0.954962,1.000000,1.0,1.0


In [21]:
# Aggregate metrics
# CHANGE: now includes precision_at_1 and precision_at_5;
#         computes means from DataFrame columns for consistency.

metric_cols = [
    "rouge1_f", "rouge2_f", "rougeL_f",
    "bleu", "bertscore_f", "faithfulness",
    "precision_at_1", "precision_at_5"
]

aggregate = {f"mean_{col}": float(np.mean(df[col])) for col in metric_cols}

print("\nOverall Evaluation Results:")
print("-" * 45)
for k, v in aggregate.items():
    print(f"{k:30s}: {v:.4f}")


Overall Evaluation Results:
---------------------------------------------
mean_rouge1_f                 : 0.3570
mean_rouge2_f                 : 0.1560
mean_rougeL_f                 : 0.2992
mean_bleu                     : 0.0739
mean_bertscore_f              : 0.8927
mean_faithfulness             : 0.7797
mean_precision_at_1           : 0.8462
mean_precision_at_5           : 1.0000


In [22]:
# NEW: Per-difficulty and per-question-type breakdown
#
# Shows where the system excels and where it struggles —
# useful for the oral presentation and written analysis sections.

print("\nBreakdown by difficulty:")
difficulty_df = df.groupby("difficulty")[metric_cols].mean().round(3)
print(difficulty_df.to_string())

print("\nBreakdown by question type:")
type_df = df.groupby("question_type")[metric_cols].mean().round(3)
print(type_df.to_string())


Breakdown by difficulty:
            rouge1_f  rouge2_f  rougeL_f   bleu  bertscore_f  faithfulness  precision_at_1  precision_at_5
difficulty                                                                                                
easy           0.446     0.232     0.397  0.134        0.906         0.816            0.80             1.0
hard           0.252     0.112     0.213  0.030        0.882         0.785            0.75             1.0
medium         0.351     0.105     0.263  0.043        0.887         0.730            1.00             1.0

Breakdown by question type:
               rouge1_f  rouge2_f  rougeL_f   bleu  bertscore_f  faithfulness  precision_at_1  precision_at_5
question_type                                                                                                
comparative       0.287     0.096     0.232  0.030        0.886         0.745           0.667             1.0
factual           0.470     0.255     0.430  0.155        0.909         0.849   

In [23]:
# NEW: Baseline comparison table
#
# Replaces the prose-only markdown cell from the original.
# Baseline = dense retrieval only (no BM25, no reranker),
# original all-MiniLM-L6-v2 embeddings, 80-token generation limit.
#
# These scores are from the original notebook run.
# Update BASELINE_SCORES if you have fresh baseline numbers.

BASELINE_SCORES = {
    "mean_rouge1_f":       0.4047,
    "mean_rouge2_f":       0.1723,
    "mean_rougeL_f":       0.3311,
    "mean_bleu":           0.0583,
    "mean_bertscore_f":    0.9014,
    "mean_faithfulness":   0.7908,
    "mean_precision_at_1": None,   # not measured in baseline
    "mean_precision_at_5": None,
}

print("\n" + "=" * 65)
print(f"{'Metric':<30} {'Baseline':>10} {'Improved':>10} {'Delta':>10}")
print("=" * 65)

for k in aggregate:
    improved = aggregate[k]
    baseline = BASELINE_SCORES.get(k)
    if baseline is not None:
        delta = improved - baseline
        sign  = "+" if delta >= 0 else ""
        print(f"{k:<30} {baseline:>10.4f} {improved:>10.4f} {sign+f'{delta:.4f}':>10}")
    else:
        print(f"{k:<30} {'N/A':>10} {improved:>10.4f} {'NEW':>10}")

print("=" * 65)


Metric                           Baseline   Improved      Delta
mean_rouge1_f                      0.4047     0.3570    -0.0477
mean_rouge2_f                      0.1723     0.1560    -0.0163
mean_rougeL_f                      0.3311     0.2992    -0.0319
mean_bleu                          0.0583     0.0739    +0.0156
mean_bertscore_f                   0.9014     0.8927    -0.0087
mean_faithfulness                  0.7908     0.7797    -0.0111
mean_precision_at_1                   N/A     0.8462        NEW
mean_precision_at_5                   N/A     1.0000        NEW


In [24]:
# NEW: Retrieval audit — did we fetch the right document for each query?
#
# Flags any query where the ground-truth source doc was not retrieved
# in the top-5, making it easy to identify retrieval failures.

print("\nRetrieval audit (did we fetch the right document?):")
print(f"{'Query ID':<10} {'P@1':>5} {'P@5':>5} {'Source doc':<30} {'Top retrieved doc'}")
print("-" * 80)

for i, qid in enumerate(common_ids):
    ret_ids    = retrieved_doc_ids[i]
    answer_src = gold_map[qid]["answer_source"]
    p1  = df.loc[df["query_id"] == qid, "precision_at_1"].values[0]
    p5  = df.loc[df["query_id"] == qid, "precision_at_5"].values[0]
    top = ret_ids[0] if ret_ids else "NONE"
    flag = "" if p5 == 1.0 else " <- MISSED"
    print(f"{qid:<10} {p1:>5.0f} {p5:>5.0f} {answer_src:<30} {top}{flag}")


Retrieval audit (did we fetch the right document?):
Query ID     P@1   P@5 Source doc                     Top retrieved doc
--------------------------------------------------------------------------------
sa_000         1     1 wiki_biryani                   wiki_biryani
sa_001         0     1 wiki_biryani                   wiki_hyderabadi_biryani
sa_002         0     1 wiki_naan                      wiki_roti
sa_003         1     1 wiki_dosa                      wiki_dosa
sa_004         1     1 wiki_samosa                    wiki_samosa
sa_005         1     1 wiki_cardamom                  wiki_cardamom
sa_006         1     1 wiki_sri_lankan_cuisine        wiki_sri_lankan_cuisine
sa_007         1     1 wiki_curry                     wiki_curry
sa_008         1     1 wiki_pakistani_cuisine         wiki_pakistani_cuisine
sa_009         1     1 wiki_butter_chicken            wiki_butter_chicken
sa_010         1     1 wiki_tandoori_chicken          wiki_tandoori_chicken
sa_011         1 

In [25]:
# Save evaluation results
# CHANGE: output JSON now includes a baseline_comparison section
#         with per-metric deltas, enabling automated comparison scripts.

eval_results = {
    "per_item":  per_item,
    "aggregate": aggregate,
    "baseline_comparison": {
        "baseline": BASELINE_SCORES,
        "improved": aggregate,
        "delta": {
            k: round(aggregate[k] - BASELINE_SCORES[k], 4)
            for k in aggregate
            if BASELINE_SCORES.get(k) is not None
        }
    }
}

with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)

print(f"Saved evaluation results to {eval_path}")

Saved evaluation results to evaluation_results.json


In [26]:
# Show aggregate results clearly (unchanged)

print("Evaluation Results (aggregate):")
for k, v in aggregate.items():
    print(f"{k:30s}: {v:.4f}")

Evaluation Results (aggregate):
mean_rouge1_f                 : 0.3570
mean_rouge2_f                 : 0.1560
mean_rougeL_f                 : 0.2992
mean_bleu                     : 0.0739
mean_bertscore_f              : 0.8927
mean_faithfulness             : 0.7797
mean_precision_at_1           : 0.8462
mean_precision_at_5           : 1.0000
